# Práctica 9 — Análisis de transcriptómica de célula única (scRNA-seq): PBMC 3k

**Máster en Bioinformática — Módulo de Transcriptómica**
**Autor:** Víctor Sánchez-Arévalo · [github.com/victorsanchezarevalo/transcriptomica-master](https://github.com/victorsanchezarevalo/transcriptomica-master)

---

En las prácticas 1–8 trabajamos con **RNA-seq a granel (bulk)**: cada muestra es la media de millones de células. En esta práctica damos el salto a la **resolución de célula única**: mediremos el transcriptoma de **2.700 células individuales** y descubriremos los tipos celulares que componen una muestra de sangre periférica.

**Dataset:** PBMC 3k de 10x Genomics — células mononucleares de sangre periférica de un donante sano, secuenciadas con la plataforma Chromium (química v1) y procesadas con Cell Ranger. Es el dataset de referencia del ecosistema [scanpy](https://scanpy.readthedocs.io).

**Objetivos de aprendizaje:**

1. Entender la estructura de una matriz de conteos de célula única (objeto `AnnData`) y sus diferencias con bulk.
2. Aplicar control de calidad **por célula** (conteos, genes detectados, % mitocondrial) y **por gen**.
3. Normalizar por tamaño de librería, transformar en log y seleccionar genes altamente variables (HVG).
4. Construir el flujo estándar de análisis: PCA → grafo de vecinos → UMAP → clustering de Leiden.
5. Identificar marcadores diferenciales por clúster y **anotar los tipos celulares** con marcadores canónicos.

**Flujo de trabajo:**

```
matriz de conteos → QC → normalización → HVG → PCA → vecinos → UMAP → Leiden → marcadores → anotación
```

> **Tiempo estimado:** una sesión de 2 horas.

## 0. Instalación y carga de librerías

[scanpy](https://scanpy.readthedocs.io) (*Single-Cell Analysis in Python*; Wolf *et al.*, Genome Biology 2018) es la librería estándar del ecosistema [scverse](https://scverse.org) para análisis de célula única. `leidenalg` implementa el algoritmo de clustering de Leiden que usaremos al final.

In [ ]:
!pip install -q scanpy leidenalg

In [ ]:
import importlib.metadata

import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1  # mensajes informativos de progreso

for pkg in ["scanpy", "anndata", "numpy", "pandas"]:
    print(f"{pkg}: {importlib.metadata.version(pkg)}")

## 1. Descarga y carga de los datos

La salida de Cell Ranger (el software de 10x Genomics) es un directorio con tres ficheros:

- `matrix.mtx`: matriz de conteos en formato disperso [Matrix Market](https://math.nist.gov/MatrixMarket/formats.html) — solo se guardan las entradas distintas de cero (¡la matriz tiene >95 % de ceros!).
- `barcodes.tsv`: los *códigos de barras* celulares (identifican cada célula).
- `genes.tsv`: los identificadores de los genes.

Descargamos la matriz filtrada (`filtered_gene_bc_matrices`), que solo contiene los barcodes que Cell Ranger identificó como células reales.

In [ ]:
import os, tarfile, urllib.request
from pathlib import Path

DATA_DIR = Path("datos_pbmc3k")
DATA_DIR.mkdir(exist_ok=True)

URL = "https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"
URL_ESPEJO = "https://raw.githubusercontent.com/victorsanchezarevalo/transcriptomica-master/main/practicas/09-scrna-seq-pbmc/datos/pbmc3k_filtered_gene_bc_matrices.tar.gz"
TAR_PATH = DATA_DIR / "pbmc3k.tar.gz"

def descargar(url, destino):
    """Descarga con cabecera de navegador (el CDN de 10x rechaza el agente por defecto de Python)."""
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=180) as r, open(destino, "wb") as f:
        f.write(r.read())

if not TAR_PATH.exists():
    try:
        print("Descargando de 10x Genomics (~7,6 MB)...")
        descargar(URL, TAR_PATH)
    except Exception as e:
        print(f"Fallo la descarga desde 10x: {e}")
        print("Probando espejo en el repositorio del curso...")
        try:
            descargar(URL_ESPEJO, TAR_PATH)
        except Exception as e2:
            print(f"Tambien fallo: {e2}")
            print("-> Sube manualmente 'pbmc3k_filtered_gene_bc_matrices.tar.gz' a la carpeta", DATA_DIR)
            raise

with tarfile.open(TAR_PATH) as tf:
    tf.extractall(DATA_DIR)

print("Contenido:", sorted(p.name for p in DATA_DIR.iterdir()))
print("Matriz:", sorted(p.name for p in (DATA_DIR / "filtered_gene_bc_matrices" / "hg19").iterdir()))

Cargamos la matriz con `sc.read_10x_mtx`. Usamos los **símbolos génicos** como nombres de variable y los hacemos únicos (algunos símbolos aparecen duplicados porque hay varios transcritos/IDs de Ensembl con el mismo símbolo).

In [ ]:
adata = sc.read_10x_mtx(
    DATA_DIR / "filtered_gene_bc_matrices" / "hg19",
    var_names="gene_symbols",
    cache=False,
)
adata.var_names_make_unique()
adata

## 2. El objeto `AnnData`: anatomía de un dataset de célula única

`AnnData` (Annotated Data) es la estructura central del ecosistema scverse. Es el equivalente a la "matriz de expresión + metadatos" que usamos en bulk, pero todo en un solo objeto:

| Componente | Contenido | Dimensión |
|---|---|---|
| `adata.X` | matriz de conteos (dispersa) | células × genes |
| `adata.obs` | metadatos **por célula** (*observations*) | nº células |
| `adata.var` | metadatos **por gen** (*variables*) | nº genes |
| `adata.layers` | matrices alternativas (p. ej. conteos crudos) | células × genes |
| `adata.obsm` | matrices por célula (PCA, UMAP) | nº células × k |

Inspeccionemos sus partes.

In [ ]:
print("Dimensiones:", adata.shape[0], "células x", adata.shape[1], "genes")
print("\nobs (metadatos por célula):", list(adata.obs.columns))
print("var (metadatos por gen):", list(adata.var.columns))
adata.var.head()

In [ ]:
# ¿Qué fracción de la matriz son ceros?
densidad = adata.X.nnz / (adata.shape[0] * adata.shape[1])
print(f"Entradas distintas de cero: {adata.X.nnz:,}")
print(f"Densidad: {densidad:.2%}  ->  {1 - densidad:.2%} de ceros")
print(f"Mediana de conteos por célula: {int(np.median(adata.X.sum(axis=1))):,}")

> **Pregunta 1.** Compara esta densidad con una matriz de bulk RNA-seq. ¿Por qué en célula única casi todas las entradas son cero? ¿Qué implicaciones tiene para el ruido y para cómo almacenamos la matriz?

## 3. Control de calidad (QC)

En scRNA-seq el QC se hace **por célula**: queremos eliminar células rotas, vacías o dobletes antes de analizar. Tres métricas clásicas:

1. **`n_genes_by_counts`**: nº de genes detectados por célula. Muy bajo → célula vacía/rota; muy alto → posible **doblete** (dos células en una gota).
2. **`total_counts`**: tamaño de librería de la célula.
3. **`pct_counts_mt`**: % de conteos de genes **mitocondriales**. Las células moribundas pierden el RNA citoplasmático por la membrana rota, pero conservan el mitocondrial → %MT alto indica baja calidad.

Calculamos las métricas y las visualizamos.

In [ ]:
# Los genes mitocondriales humanos empiezan por "MT-"
adata.var["mt"] = adata.var_names.str.startswith("MT-")
print("Genes mitocondriales detectados:", adata.var['mt'].sum())

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe()

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    multi_panel=True,
    jitter=0.4,
)

In [ ]:
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts")

**Interpretación:**

- En el primer scatter, la nube principal baja y compacta es de células sanas; la cola con %MT alto son células de mala calidad.
- En el segundo, la relación conteos–genes es casi lineal; los puntos que se separan hacia arriba (muchos genes para sus conteos) son candidatos a dobletes.

**Umbrales** (estándar para este dataset, los del tutorial oficial de scanpy):

- ≥ 200 genes detectados por célula, y gen expresado en ≥ 3 células.
- < 2.500 genes por célula (elimina dobletes).
- < 5 % de conteos mitocondriales.

In [ ]:
n_celulas_ini, n_genes_ini = adata.shape

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[
    (adata.obs["n_genes_by_counts"] < 2500) & (adata.obs["pct_counts_mt"] < 5)
].copy()

print(f"Células: {n_celulas_ini} -> {adata.n_obs}")
print(f"Genes:   {n_genes_ini} -> {adata.n_vars}")

> **Pregunta 2.** ¿Qué porcentaje de células se ha eliminado? Si el experimento fuera de tejido tumoral disociado (más estrés celular), ¿esperarías tener que relajar o endurecer el umbral de %MT? ¿Qué riesgo tiene endurecerlo demasiado?

## 4. Normalización y transformación logarítmica

Cada célula se secuencia a una profundidad distinta (suerte de la gota, amplificación...). Para comparar células entre sí normalizamos **por tamaño de librería**: escalamos los conteos de cada célula para que sumen 10.000 (análogo a CPM en bulk) y aplicamos `log1p` (= log(x+1)) para estabilizar la varianza.

Antes, guardamos los **conteos crudos** en una capa (`layers["counts"]`): los necesitaremos si queremos aplicar métodos que modelan conteos (como PyDESeq2, que usaremos en la práctica 10).

In [ ]:
# Guardamos los conteos crudos para usos posteriores
adata.layers["counts"] = adata.X.copy()

# Normalización por tamaño de librería + log
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Guardamos la matriz normalizada completa en .raw (la usaremos para marcadores)
adata.raw = adata
adata

## 5. Selección de genes altamente variables (HVG)

La mayoría de los 13.714 genes no aportan información para distinguir tipos celulares (no se expresan, o se expresan igual en todas partes) y solo añaden ruido y coste computacional. Nos quedamos con los **genes altamente variables**: aquellos cuya variabilidad entre células supera la esperada por su nivel de expresión.

> **Conexión con la práctica 6 (WGCNA):** allí seleccionamos los 5.000 genes más variables entre muestras de bulk por el mismo motivo. Aquí la selección es más sofisticada: modela la relación media–dispersión propia del recuento.

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
print("Genes altamente variables:", int(adata.var["highly_variable"].sum()))
sc.pl.highly_variable_genes(adata)

A partir de aquí trabajamos **solo con los HVG** para los pasos de reducción de dimensionalidad. Además:

- `sc.pp.regress_out`: regresa fuera dos fuentes de variación técnica (`total_counts` residual y `pct_counts_mt`).
- `sc.pp.scale`: escala cada gen a media 0 y varianza 1 (con techo en 10), para que los genes muy expresados no dominen el PCA.

In [ ]:
adata = adata[:, adata.var["highly_variable"]].copy()
sc.pp.regress_out(adata, ["total_counts", "pct_counts_mt"])
sc.pp.scale(adata, max_value=10)
print("Matriz de trabajo:", adata.shape)

## 6. Reducción de dimensionalidad: PCA

Igual que en bulk (práctica 2), el PCA resume la variación transcriptómica en unas pocas componentes. Aquí el PCA no es el fin, sino el medio: el grafo de vecinos y el UMAP se construirán sobre las componentes principales, no sobre los genes.

In [ ]:
sc.tl.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, log=True)

> **Pregunta 3.** Mirando la varianza explicada, ¿cuántas componentes principales retienen señal sustancial? Usaremos 40 en el siguiente paso: ¿te parece razonable?

## 7. Grafo de vecinos y UMAP

Para visualizar 2.700 células en 2D usamos **UMAP**, que se construye sobre un **grafo de k vecinos más próximos** en el espacio del PCA:

1. `sc.pp.neighbors`: cada célula se conecta con sus 10 células más similares (en 40 PCs).
2. `sc.tl.umap`: incrusta el grafo en 2D preservando la estructura local.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)
sc.pl.umap(adata)

Ya se ven islas claras de células. ¿Son tipos celulares reales? Para comprobarlo necesitamos **agrupar** las células formalmente.

## 8. Clustering con el algoritmo de Leiden

Sobre el mismo grafo de vecinos aplicamos **detección de comunidades de Leiden** (Traag *et al.*, Scientific Reports 2019; sucesor de Louvain): busca particiones del grafo con muchas conexiones dentro de cada grupo y pocas entre grupos.

El parámetro `resolution` controla el número de comunidades: valores mayores → más clústeres (más resolución). Con 0,8 obtenemos una partición que reproduce bien los tipos principales de PBMC.

In [ ]:
sc.tl.leiden(adata, resolution=0.8, key_added="leiden")
print(adata.obs["leiden"].value_counts().sort_index())
sc.pl.umap(adata, color="leiden", legend_loc="on data")

## 9. Genes marcadores por clúster

¿Qué hace distinto a cada clúster? Comparamos cada clúster contra el resto con el test de **Wilcoxon** (robusto para datos de célula única, recomendado por scanpy) sobre la matriz normalizada completa guardada en `.raw`.

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon", use_raw=True)
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False)

In [ ]:
# Tabla con los 5 mejores marcadores de cada clúster
marcadores = sc.get.rank_genes_groups_df(adata, group=None)
top5 = marcadores.groupby("group").head(5)
top5[["group", "names", "logfoldchanges", "pvals_adj"]].reset_index(drop=True)

## 10. Anotación de tipos celulares con marcadores canónicos

Los nombres de los marcadores ya nos suenan: `NKG7` (NK), `CD79A` (células B), `LYZ`/`S100A9` (monocitos)... Formalicemos la anotación con un panel de **marcadores canónicos** de tipos de PBMC:

| Tipo celular | Marcadores |
|---|---|
| T CD4 | `IL7R`, `CCR7` |
| T CD8 | `CD8A`, `CD8B` |
| NK | `GNLY`, `NKG7` |
| B | `MS4A1` (CD20), `CD79A` |
| Monocitos CD14+ | `CD14`, `LYZ`, `S100A9` |
| Monocitos FCGR3A+ (CD16+) | `FCGR3A`, `MS4A7` |
| Dendríticas | `FCER1A`, `CST3` |
| Megacariocitos | `PPBP`, `PF4` |

In [ ]:
marcadores_canonicos = {
    "T CD4": ["IL7R", "CCR7"],
    "T CD8": ["CD8A", "CD8B"],
    "NK": ["GNLY", "NKG7"],
    "B": ["MS4A1", "CD79A"],
    "Mono CD14+": ["CD14", "LYZ", "S100A9"],
    "Mono FCGR3A+": ["FCGR3A", "MS4A7"],
    "Dendríticas": ["FCER1A", "CST3"],
    "Megacariocitos": ["PPBP", "PF4"],
}

sc.pl.dotplot(adata, marcadores_canonicos, groupby="leiden")

**Ejercicio 1.** Mirando el dotplot, asigna a cada clúster de Leiden (0–6) un tipo celular y completa el diccionario de abajo. Comprueba tu respuesta con los marcadores diferenciales de la sección 9.

In [ ]:
# Completa el diccionario clúster -> tipo celular (solución más abajo)
anotacion = {
    "0": "T CD4",
    "1": "Monocitos CD14+",
    "2": "NK",
    "3": "B",
    "4": "Monocitos FCGR3A+",
    "5": "Dendríticas",
    "6": "Megacariocitos",
}

adata.obs["cell_type"] = adata.obs["leiden"].map(anotacion).astype("category")
print(adata.obs["cell_type"].value_counts())
sc.pl.umap(adata, color="cell_type", legend_loc="on data")

## 11. (Opcional) Explorar la resolución del clustering

> **Ejercicio 2.** Repite el clustering con `resolution=0.4` y con `resolution=1.5` y compara los UMAP. ¿Qué clúster se fusiona al bajar la resolución? ¿Qué tipo celular se sobrefragmenta al subirla? ¿Cuál elegirías para presentar estos datos y por qué?

In [ ]:
# Tu código aquí: repite sc.tl.leiden con otra resolución y dibuja el UMAP
# sc.tl.leiden(adata, resolution=0.4, key_added="leiden_04")
# sc.pl.umap(adata, color=["leiden", "leiden_04"])

## 12. Guardar el resultado

Guardamos el objeto `AnnData` procesado en formato `.h5ad` (HDF5): contiene la matriz, los metadatos y todas las representaciones calculadas. Lo comprimimos para descargarlo.

In [ ]:
import zipfile

adata.write("pbmc3k_procesado.h5ad")

with zipfile.ZipFile("pbmc3k_procesado.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("pbmc3k_procesado.h5ad")

print("Guardado pbmc3k_procesado.zip")

In [ ]:
# Descargar a tu ordenador
from google.colab import files
files.download("pbmc3k_procesado.zip")

## Resumen

| Paso | Función | Idea clave |
|---|---|---|
| Carga | `sc.read_10x_mtx` | matriz dispersa células × genes |
| QC | `sc.pp.calculate_qc_metrics`, `filter_cells/genes` | eliminar células rotas y dobletes |
| Normalización | `normalize_total` + `log1p` | corregir tamaño de librería |
| HVG | `highly_variable_genes` | quedarse con la señal |
| PCA | `sc.tl.pca` | compresión con ruido reducido |
| Vecinos + UMAP | `sc.pp.neighbors`, `sc.tl.umap` | grafo → mapa 2D |
| Leiden | `sc.tl.leiden` | comunidades en el grafo |
| Marcadores | `sc.tl.rank_genes_groups` | qué define cada clúster |
| Anotación | marcadores canónicos | de clúster a biología |

**Puentes con el resto del curso:**

- Con la **práctica 8 (deconvolución)**: ahí estimábamos las proporciones de tipos celulares en bulk usando LM22; aquí hemos *visto* esos tipos directamente. La deconvolución es, en esencia, intentar recuperar esta información sin secuenciar célula a célula.
- Con la **práctica 10**: ¿y si tengo varias muestras y condiciones? Necesitaremos integrar muestras (efecto batch) y hacer el análisis diferencial respetando las réplicas biológicas (pseudobulk).

> **Pregunta final.** El dataset tiene una única muestra de un donante sano. Si encontramos que el 40 % de sus células son T CD4, ¿podemos concluir algo sobre la población general? ¿Qué necesitaríamos?